# 🧪 Experimentation Notebook
## Customer Churn ANN — End-to-End Training & Evaluation

This notebook runs the full pipeline: data ingestion → preprocessing → model training → evaluation → registry.

```mermaid
flowchart LR
    A[Raw CSV] --> B[ChurnDataPipeline]
    B --> C[DataSplit]
    C --> D[ChurnANN.build]
    D --> E[ChurnModelTrainer.train]
    E --> F[ModelEvaluator.evaluate]
    F --> G[ModelRegistry.register]
```


In [ ]:
import sys, warnings, logging
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
print("Environment ready ✓")


## 1. Data Pipeline

In [ ]:
from src.data.pipeline import ChurnDataPipeline, PipelineConfig

config = PipelineConfig(
    test_size=0.20,
    val_size=0.10,
    random_state=42,
    stratify=True,
    save_artifacts=True,
    artifacts_dir='../artifacts/pipeline',
)

pipeline = ChurnDataPipeline(config)
split = pipeline.run('../data/raw/ChurnPrediction.csv')

print(f"Train : {split.train_size:,} samples  | class dist: {split.class_distribution('train')}")
print(f"Val   : {split.val_size:,} samples  | class dist: {split.class_distribution('val')}")
print(f"Test  : {split.test_size:,} samples  | class dist: {split.class_distribution('test')}")
print(f"Data hash (SHA-256): {split.data_hash[:16]}...")


## 2. EDA

In [ ]:
df = pd.read_csv('../data/raw/ChurnPrediction.csv', index_col='RowNumber')
df = df.drop(columns=['CustomerId','Surname'])
df['Geography'] = df['Geography'].map({'France':0,'Germany':1,'Spain':2})
df['Gender'] = df['Gender'].map({'Female':0,'Male':1})

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.ravel()
for i, col in enumerate(df.columns):
    df[col].hist(ax=axes[i], bins=25, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')
fig.suptitle('Feature Distributions', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()
print(f"Class balance — Not Churn: {(df.Exited==0).sum()} | Churn: {(df.Exited==1).sum()}")


In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Model Training — Baseline

In [ ]:
from src.models.ann import ChurnANN, baseline_config
from src.training.trainer import ChurnModelTrainer, TrainingConfig

model_config = baseline_config(input_dim=split.X_train.shape[1])
model = ChurnANN(model_config).build()
model.summary()


In [ ]:
trainer = ChurnModelTrainer(TrainingConfig(
    epochs=100,
    batch_size=32,
    early_stopping_patience=15,
    checkpoint_dir='../artifacts/checkpoints',
    tensorboard_log_dir='../artifacts/tensorboard',
    enable_mlflow=False,
    verbose=1,
))

result = trainer.train(
    model,
    split.X_train, split.y_train,
    split.X_val,   split.y_val,
    model_config=model_config.to_dict(),
)

print(f"\nBest epoch : {result.best_epoch}")
print(f"Best val loss: {result.best_val_loss:.4f}")
print(f"Best val AUC : {result.best_val_auc:.4f}")
print(f"Train time   : {result.train_duration_seconds:.1f}s")


## 4. Training Curves

In [ ]:
history = result.history
epochs_range = range(1, len(history['loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history['loss'], label='Train Loss', color='steelblue')
ax1.plot(epochs_range, history['val_loss'], label='Val Loss', color='tomato')
ax1.axvline(result.best_epoch, linestyle='--', color='gray', alpha=0.6, label=f'Best epoch={result.best_epoch}')
ax1.set_title('Loss Curves'); ax1.legend(); ax1.set_xlabel('Epoch')

auc_key = [k for k in history if k.startswith('val_auc')]
if auc_key:
    ax2.plot(epochs_range, history['auc'], label='Train AUC', color='steelblue')
    ax2.plot(epochs_range, history[auc_key[0]], label='Val AUC', color='tomato')
    ax2.axvline(result.best_epoch, linestyle='--', color='gray', alpha=0.6)
    ax2.set_title('AUC Curves'); ax2.legend(); ax2.set_xlabel('Epoch')

fig.suptitle('Training Diagnostics', fontsize=13, fontweight='bold')
fig.tight_layout(); plt.show()


## 5. Full Evaluation Suite

In [ ]:
from src.evaluation.evaluator import ModelEvaluator

evaluator = ModelEvaluator(output_dir='../artifacts/evaluation', threshold=0.5)
eval_result = evaluator.evaluate(model, split.X_test, split.y_test, split='test')
evaluator.print_report(eval_result)

y_proba = model.predict(split.X_test, verbose=0).ravel()
evaluator.plot_all(eval_result, split.y_test, y_proba)
print(f"Plots saved to ../artifacts/evaluation/")


## 6. Threshold Analysis

In [ ]:
thresholds = np.linspace(0.1, 0.9, 80)
results_by_thresh = []
from sklearn.metrics import f1_score, precision_score, recall_score

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    results_by_thresh.append({
        'threshold': t,
        'f1': f1_score(split.y_test, y_pred_t, zero_division=0),
        'precision': precision_score(split.y_test, y_pred_t, zero_division=0),
        'recall': recall_score(split.y_test, y_pred_t, zero_division=0),
    })

df_thresh = pd.DataFrame(results_by_thresh)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_thresh.threshold, df_thresh.f1, label='F1', linewidth=2)
ax.plot(df_thresh.threshold, df_thresh.precision, label='Precision', linewidth=2)
ax.plot(df_thresh.threshold, df_thresh.recall, label='Recall', linewidth=2)
ax.axvline(eval_result.optimal_threshold_f1, linestyle='--', color='black',
           label=f'Optimal F1 threshold = {eval_result.optimal_threshold_f1:.3f}')
ax.set_xlabel('Decision Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Sensitivity Analysis')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Optimal F1 threshold : {eval_result.optimal_threshold_f1:.3f}")
print(f"Threshold for 90% recall: {eval_result.optimal_threshold_recall_90p}")


## 7. Model Registry

In [ ]:
from src.utils.model_registry import ModelRegistry

registry = ModelRegistry(root='../artifacts/model_registry')
entry = registry.register(
    name='churn-ann',
    model_path=result.model_path,
    metrics={
        'val_auc': result.best_val_auc,
        'val_loss': result.best_val_loss,
        'test_roc_auc': eval_result.roc_auc,
        'test_f1': eval_result.f1,
        'test_accuracy': eval_result.accuracy,
    },
    tags={'preset': 'baseline', 'notebook': 'experimentation'},
    description='Baseline 6-6 ANN, no dropout',
)
registry.promote(entry.model_id, stage='champion')
registry.print_registry()
print("Champion model registered ✓")
